# RF-DETR Multi-Class Model Training (Clean Data Pipeline)
### Precision-Driven Training Pipeline with Step A, Step B & Step C Diagnostics

This notebook fine-tunes an RF-DETR model exclusively on **clean, audited, and stratified multi-class data** with a direct focus on **Metric Precision**:
- **Metric Precision Optimization**: Tracks and trains for operational **Precision** (TP / (TP + FP)), **Recall**, and **F1-Score** alongside mAP at every epoch, prioritizing false positive suppression.
- **Precision-Aware Checkpointing**: Automatically selects and exports the checkpoint with the highest validation Precision (`best_model_precision.pth`).
- **Zero-Anomaly Policy**: Strictly excludes any anomaly detected and logged in `02_anomalies/anomalies_manifest.json`.
- **Step A (Loss Component Breakdown)**: Separately logs classification cross-entropy (`loss_ce`), bounding box L1 (`loss_bbox`), and generalized IoU (`loss_giou`) to isolate query calibration and box regression dynamics.
- **Step B (Label & Category Alignment)**: Automatic pre-flight verification of contiguous 0-indexing, category coverage across splits, class imbalance, and bounding box validity.
- **Step C (Regularization & Frozen Backbone)**: Freezes DINOv2 ViT backbone (`FREEZE_BACKBONE = True`) to accelerate training and prevent catastrophic forgetting on domain-specific data.
- **Asynchronous Prefetching & AMP**: High-throughput DataLoaders with pinned memory, OpenCV C++ decoding, and PyTorch Automatic Mixed Precision (`torch.amp.autocast`).

In [ ]:
# STEP 0 - Environment Dependencies
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install --no-cache-dir opencv-python-headless==4.9.0.80
%pip install python-dotenv


In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# ==============================================================================
# CELL 2 - MASTER CONFIGURATION & HYPERPARAMETERS
# Clean Data Pipeline Configuration: Precision-Driven Metric Training
# ==============================================================================
from pathlib import Path

# --- Pipeline Identification ---
PIPELINE_NAME = "RF-DETR Clean Multi-Class Detection (Precision Optimized)"
MODEL_VERSION = "v1.1.0-precision-clean"
RUN_ID = f"rfdetr_precision_{time.strftime('%Y%m%d_%H%M%S')}"

# --- Directory Paths ---
# Resilient pipeline directory resolver
BASE_DIR = next((p.resolve() for p in [Path("."), Path(".."), Path("../..")] if (p / "03_dataset_split").exists()), Path(".").resolve())
SPLIT_DIR = next((p for p in [Path("dataset_stratified"), Path("03_dataset_split/dataset_stratified"), Path("../dataset_stratified")] if p.exists()), Path("dataset_stratified"))
ANOMALIES_MANIFEST_PATH = next((p for p in [Path("anomalies_manifest.json"), Path("02_anomalies/anomalies_manifest.json"), Path("../02_anomalies/anomalies_manifest.json")] if p.exists()), Path("anomalies_manifest.json"))

TRAIN_ANN = str(SPLIT_DIR / "train" / "train_annotations.json")
VAL_ANN = str(SPLIT_DIR / "val" / "val_annotations.json")
TEST_ANN = str(SPLIT_DIR / "test" / "test_annotations.json")
IMAGES_DIR = str(SPLIT_DIR / "images")

# Fallback paths if split directory is located at root
if not Path(TRAIN_ANN).exists():
    ALT_SPLIT = BASE_DIR / "dataset_stratified"
    if ALT_SPLIT.exists():
        SPLIT_DIR = ALT_SPLIT
        TRAIN_ANN = str(SPLIT_DIR / "train" / "train_annotations.json")
        VAL_ANN = str(SPLIT_DIR / "val" / "val_annotations.json")
        TEST_ANN = str(SPLIT_DIR / "test" / "test_annotations.json")
        IMAGES_DIR = str(SPLIT_DIR / "images")

OUTPUT_DIR = str(BASE_DIR / "output" / RUN_ID)
CHECKPOINT_DIR = str(BASE_DIR / "checkpoints")
LOG_DIR = str(BASE_DIR / "logs")
INFERENCE_OUTPUT_DIR = str(BASE_DIR / "inference_outputs")
FINAL_MODEL_DIR = str(BASE_DIR / "model")

# --- Model & Backbone Configuration ---
MODEL_SIZE = "base"                        # 'base' or 'small'
PRETRAINED_WEIGHTS = "rf-detr-base.pth"     # Base model checkpoint
RESOLUTION = 560                            # Spatial resolution (must be divisible by divisor)
FREEZE_BACKBONE = True                      # Step C: Freeze DINOv2 ViT backbone

# Resolution Divisor Sanity Check
def calculate_valid_resolution(base_res: int, model_type: str = "base") -> int:
    divisor = 56 if model_type == "base" else 32
    return round(base_res / divisor) * divisor

RESOLUTION = calculate_valid_resolution(RESOLUTION, MODEL_SIZE)

# --- Training Hyperparameters ---
EPOCHS = 10
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4                       # Effective Batch Size = 4 * 4 = 16
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUMULATION
LEARNING_RATE = 1e-4
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
LR_DROP_EPOCHS = [7, 9]
STEPS_PER_EPOCH = None                      # None for full epoch

# --- Precision-Driven Metric & Checkpointing Configuration ---
MONITOR_METRIC = "precision"                # Checkpoint selection metric: 'precision', 'f1', 'map_50', or 'composite'
TARGET_PRECISION_CONF_THRESH = 0.50         # Confidence threshold for computing operational precision
TARGET_PRECISION_IOU_THRESH = 0.50          # IoU threshold for True Positive box matching
OPTIMIZE_FOR_PRECISION = True               # Save dedicated best precision checkpoint
COMPOSITE_PRECISION_WEIGHT = 0.6            # Weight for precision when MONITOR_METRIC == 'composite'

# --- Step A: Loss Component Coefficients (Tuned for Precision & FP Suppression) ---
CE_LOSS_COEF = 2.5                          # Classification Cross-Entropy Loss (heightened to penalize label error)
BBOX_LOSS_COEF = 5.0                        # Bounding Box L1 Loss
GIOU_LOSS_COEF = 2.0                        # Generalized IoU Loss
EOS_COEF = 0.15                             # No-object (background) query penalty to suppress false positives

# --- Step C: Early Stopping & Checkpointing ---
EARLY_STOPPING_PATIENCE = 5                 # Epochs without validation improvement
MIN_DELTA = 0.001                           # Minimum required validation improvement
RESUME_TRAINING = False
RESUME_CHECKPOINT_PATH = None

# --- DataLoader Settings ---
NUM_WORKERS = min(4, os.cpu_count() or 1)
PIN_MEMORY = torch.cuda.is_available()
PERSISTENT_WORKERS = (NUM_WORKERS > 0)
USE_AUGMENTATION = True                     # Regularization augmentations

# --- Evaluation & Inference ---
CONFIDENCE_THRESHOLD = 0.50
IOU_THRESHOLD = 0.50
MAP_EVAL_THRESHOLD = 0.05
MAX_PREVIEW_IMAGES = 6

print(f"Loaded Master Configuration for Precision-Driven Pipeline:")
print(f"   - Split Directory:            {SPLIT_DIR}")
print(f"   - Monitor Metric:             {MONITOR_METRIC.upper()} (IoU={TARGET_PRECISION_IOU_THRESH}, Conf={TARGET_PRECISION_CONF_THRESH})")
print(f"   - Optimize for Precision:     {OPTIMIZE_FOR_PRECISION}")
print(f"   - Resolution:                 {RESOLUTION}x{RESOLUTION}")
print(f"   - Effective Batch Size:       {EFFECTIVE_BATCH_SIZE} (Batch={BATCH_SIZE} x Accum={GRAD_ACCUMULATION})")
print(f"   - Freeze Backbone:            {FREEZE_BACKBONE}")
print(f"   - Loss Coefficients:          CE={CE_LOSS_COEF}, BBox={BBOX_LOSS_COEF}, GIoU={GIOU_LOSS_COEF}, BG_EOS={EOS_COEF}")

# --- Bounding-Box Recalibration Configuration ---
ENABLE_BOX_AVERAGING = True                 # Weighted Box Fusion (WBF) fuses overlapping duplicate queries
ENABLE_BORDER_CALIBRATION = True            # Contracts loose query boundaries (boundary-box shrinkage)
BOX_SHRINK_FACTOR = 0.96                    # Contracts box by 4% around center to match tight label annotations
FUSION_IOU = 0.50                           # IoU threshold for clustering duplicate candidate queries


In [ ]:
# CELL 3 - Dynamic Folder Creation & File Logging
for p in [OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR, INFERENCE_OUTPUT_DIR, FINAL_MODEL_DIR]:
    os.makedirs(p, exist_ok=True)

log_file_path = os.path.join(LOG_DIR, f"{RUN_ID}.log")
file_handler = logging.FileHandler(log_file_path, mode="a")
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
logger.addHandler(file_handler)

logger.info(f"Initialized training session log at: {log_file_path}")


In [ ]:
# CELL 4 - Load Sanitized Clean Dataset & Enforce Zero-Anomaly Policy
logger.info("=" * 70)
logger.info("[CELL 4] Verifying dataset split annotations and enforcing Zero-Anomaly Policy...")
logger.info("=" * 70)

# Check existence of split files
for split_name, ann_path in [("Train", TRAIN_ANN), ("Val", VAL_ANN), ("Test", TEST_ANN)]:
    if not os.path.exists(ann_path):
        logger.warning(f"Split annotation not found at {ann_path}! Run 03_dataset_split first.")

# Load train annotations to discover categories dynamically
with open(TRAIN_ANN) as f:
    train_coco = json.load(f)

# Auto-discover categories
TARGET_CATEGORIES = sorted([c["name"] for c in train_coco["categories"]])
NUM_CLASSES = len(TARGET_CATEGORIES)
CATEGORY_TO_ID = {c["name"]: c["id"] for c in train_coco["categories"]}
ID_TO_CATEGORY = {c["id"]: c["name"] for c in train_coco["categories"]}

logger.info(f"Discovered {NUM_CLASSES} Target Categories from Clean Split:")
for cat_name, cat_id in CATEGORY_TO_ID.items():
    logger.info(f"   - [{cat_id}] {cat_name}")

# Enforce zero-anomaly policy against anomalies_manifest.json
if os.path.exists(ANOMALIES_MANIFEST_PATH):
    with open(ANOMALIES_MANIFEST_PATH) as f:
        manifest = json.load(f)
    flagged_ids = set()
    for cat_list in manifest.get("anomalies_by_category", {}).values():
        flagged_ids.update(cat_list)
    logger.info(f"Anomalies Manifest loaded: {len(flagged_ids)} total anomaly boxes flagged.")
    
    # Audit train annotations against flagged boxes
    train_box_ids = {a["id"] for a in train_coco.get("annotations", [])}
    overlap = train_box_ids.intersection(flagged_ids)
    if overlap:
        logger.warning(f"Detected {len(overlap)} anomaly boxes in train split! Filtering them out immediately.")
        train_coco["annotations"] = [a for a in train_coco["annotations"] if a["id"] not in overlap]
        with open(TRAIN_ANN, "w") as f:
            json.dump(train_coco, f, indent=2)
        logger.info(f"Train split sanitized: {len(train_coco['annotations'])} clean annotations remain.")
    else:
        logger.info(f"Zero-Anomaly Verification PASSED: 0 anomaly boxes present in clean train split.")
else:
    logger.info("No anomalies manifest found. Proceeding with clean split annotations.")


In [ ]:
# CELL 12 - Dataset Inspection & Step B Label/Category Alignment Verification
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Validate all annotation files exist
for split, path in [("train", TRAIN_ANN), ("val", VAL_ANN), ("test", TEST_ANN)]:
    assert os.path.exists(path), f"Missing annotation file for '{split}': {path}"
logger.info("All annotation files verified.")

train_info = load_coco_info(TRAIN_ANN)
val_info = load_coco_info(VAL_ANN)
test_info = load_coco_info(TEST_ANN)

# NUM_CLASSES is derived from the training annotation file
NUM_CLASSES = train_info["num_classes"]

logger.info("=" * 58)
logger.info(f"{'Split': <10} {'Images': >8} {'Annotations': >14} {'Classes': >9}")
logger.info("-" * 58)
for name, info in [("train", train_info), ("val", val_info), ("test", test_info)]:
    logger.info(f"{name: <10} {info['num_images']: >8} {info['num_anns']: >14} {info['num_classes']: >9}")
logger.info("=" * 58)
logger.info(f"Classes ({NUM_CLASSES} total): {train_info['categories']}")

# ==========================================================================
# STEP B CHECKS: Comprehensive Label & Category Alignment Diagnostic Suite
# ==========================================================================
from collections import defaultdict, Counter
def verify_label_and_category_alignment(train_path: str, val_path: str, test_path: str) -> dict:
    """Rigorous Step B diagnostic check verifying category indexing, split coverage, and box coordinates."""
    logger.info("=" * 75)
    logger.info("[STEP B CHECK] Executing Label & Category Alignment Verification Suite...")
    logger.info("=" * 75)
    
    issues = []
    splits = {"train": train_path, "val": val_path, "test": test_path}
    loaded = {}
    for sname, spath in splits.items():
        with open(spath, "r", encoding="utf-8") as f:
            loaded[sname] = json.load(f)

    # Check 1: Contiguous 0-indexing in categories
    train_cats = loaded["train"].get("categories", [])
    train_ids = sorted([c["id"] for c in train_cats])
    expected_ids = list(range(len(train_cats)))
    if train_ids != expected_ids:
        msg = f"Category IDs are not contiguous 0-indexed! Expected {expected_ids[:5]}..., got {train_ids[:5]}..."
        issues.append(msg)
        logger.error(f"   [FAIL] {msg}")
    else:
        logger.info(f"   [PASS] Contiguous 0-indexing verified: IDs {min(train_ids)} to {max(train_ids)} ({len(train_ids)} classes).")

    # Check 2: Cross-split category consistency
    train_cat_dict = {c["id"]: c["name"] for c in train_cats}
    for sname in ["val", "test"]:
        s_ids = set(c["id"] for c in loaded[sname].get("categories", []))
        diff = s_ids - set(train_ids)
        if diff:
            msg = f"{sname.capitalize()} contains category IDs not present in Train: {diff}"
            issues.append(msg)
            logger.error(f"   [FAIL] {msg}")
        else:
            logger.info(f"   [PASS] All {sname} categories align exactly with train categories.")

    # Check 3: Bounding box validity and class counts
    ann_counts = {s: defaultdict(int) for s in splits}
    degenerate_boxes = 0
    for sname, data in loaded.items():
        imgs = {im["id"]: im for im in data.get("images", [])}
        for ann in data.get("annotations", []):
            cid = ann.get("category_id")
            ann_counts[sname][cid] += 1
            bbox = ann.get("bbox", [])
            if len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0:
                degenerate_boxes += 1
                continue
            x, y, w, h = bbox
            im_meta = imgs.get(ann.get("image_id"), {})
            im_w, im_h = im_meta.get("width", 1920), im_meta.get("height", 1080)
            if x < 0 or y < 0 or (x + w) > im_w * 1.05 or (y + h) > im_h * 1.05:
                degenerate_boxes += 1

    logger.info("-" * 75)
    logger.info(f"{'ID':<5} {'Category Name':<25} {'Train Boxes':>12} {'Val Boxes':>11} {'Test Boxes':>11} {'Status':>8}")
    logger.info("-" * 75)
    for cid in train_ids:
        cname = train_cat_dict.get(cid, "Unknown")
        tr_c = ann_counts["train"][cid]
        va_c = ann_counts["val"][cid]
        te_c = ann_counts["test"][cid]
        status = "OK"
        if va_c == 0:
            status = "NO VAL!"
            issues.append(f"Category '{cname}' (ID {cid}) has 0 instances in validation split!")
        elif tr_c < 5:
            status = "LOW DATA"
            issues.append(f"Category '{cname}' (ID {cid}) has only {tr_c} train instances (<5).")
        logger.info(f"{cid:<5} {cname[:24]:<25} {tr_c:>12} {va_c:>11} {te_c:>11} {status:>8}")
    logger.info("-" * 75)

    if degenerate_boxes > 0:
        msg = f"Detected {degenerate_boxes} degenerate or out-of-bounds bounding boxes across dataset splits!"
        issues.append(msg)
        logger.warning(f"   [WARN] {msg}")
    else:
        logger.info("   [PASS] All bounding boxes verified: valid positive width/height within image limits.")

    if not issues:
        logger.info("[STEP B CHECK PASSED] Dataset labels, indexing, and splits are fully consistent.")
    else:
        logger.warning(f"[STEP B CHECK NOTICE] {len(issues)} item(s) to be aware of. Validation loss may be affected by class distribution.")
    logger.info("=" * 75)
    return {"issues": issues, "ann_counts": ann_counts}

step_b_audit = verify_label_and_category_alignment(TRAIN_ANN, VAL_ANN, TEST_ANN)

In [ ]:
# CELL 13 - Print Sample Images per Tag (In-Memory Display - No Files Saved)
from PIL import Image, ImageDraw
from IPython.display import display
from collections import defaultdict

# High-contrast color palette for distinct multi-class boxes
SAMPLE_COLORS = [
    (230, 25, 75),   # Red
    (60, 180, 75),   # Green
    (255, 225, 25),  # Yellow
    (67, 99, 216),   # Blue
    (245, 130, 49),  # Orange
    (145, 30, 180),  # Purple
    (66, 212, 244),  # Cyan
    (240, 50, 230),  # Magenta
    (191, 239, 69),  # Lime
    (250, 190, 212), # Pink
    (70, 153, 144),  # Teal
    (220, 190, 255)  # Lavender
]

def print_sample_images_per_tag(annotation_path: str, image_dir: str, max_display_width: int = 700):
    """
    Selects sample images covering each discovered category tag, annotates
    bounding boxes with colored badges in memory, and prints/displays them
    inline directly in the notebook without saving any files to disk.
    """
    print("=" * 75)
    print("[CELL 13] Printing Sample Images with Category Tags (No Disk Saving)")
    print("=" * 75)

    with open(annotation_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    img_id_map = {im["id"]: im for im in data.get("images", [])}
    
    cat_to_images = defaultdict(list)
    img_to_anns = defaultdict(list)
    for ann in data.get("annotations", []):
        cid = ann.get("category_id")
        iid = ann.get("image_id")
        cat_to_images[cid].append(iid)
        img_to_anns[iid].append(ann)

    # Match with existing local image files
    existing_cat_images = {}
    for cid, iids in cat_to_images.items():
        valid_iids = [
            iid for iid in iids
            if iid in img_id_map and os.path.exists(os.path.join(image_dir, img_id_map[iid]["file_name"]))
        ]
        if valid_iids:
            existing_cat_images[cid] = valid_iids

    if not existing_cat_images:
        print("[WARN] No valid local images found to display.")
        return

    # Display an annotated sample for every distinct category tag
    for cid in sorted(existing_cat_images.keys()):
        cat_name = categories.get(cid, str(cid))
        sample_img_id = existing_cat_images[cid][0]
        meta = img_id_map[sample_img_id]
        fname = meta["file_name"]
        img_path = os.path.join(image_dir, fname)

        img = Image.open(img_path).convert("RGB")
        im_w, im_h = img.size
        draw = ImageDraw.Draw(img)
        anns = img_to_anns.get(sample_img_id, [])

        cat_counts = defaultdict(int)
        for ann in anns:
            raw_bbox = ann.get("bbox", [])
            if len(raw_bbox) != 4:
                continue
            x, y, w, h = [float(v) for v in raw_bbox]
            ann_cid = int(ann.get("category_id", 0))
            ann_cname = str(categories.get(ann_cid, ann_cid))
            cat_counts[ann_cname] += 1
            color = SAMPLE_COLORS[ann_cid % len(SAMPLE_COLORS)]

            # Clamp box to valid dimensions
            x0 = max(0.0, min(float(im_w - 1), x))
            y0 = max(0.0, min(float(im_h - 1), y))
            x1 = max(0.0, min(float(im_w - 1), x + w))
            y1 = max(0.0, min(float(im_h - 1), y + h))

            # Draw bounding box
            draw.rectangle([x0, y0, x1, y1], outline=color, width=4)

            # Draw colored label badge with tag name
            label_text = f" {ann_cname} "
            if hasattr(draw, "textbbox"):
                tb = draw.textbbox((x0, max(0.0, y0 - 18.0)), label_text)
                draw.rectangle(tb, fill=color)
                draw.text((x0, max(0.0, y0 - 18.0)), label_text, fill=(255, 255, 255))
            else:
                draw.text((x0 + 2, max(0.0, y0 - 14.0)), label_text, fill=color)

        summary_str = ", ".join(f"'{k}': {v}" for k, v in sorted(cat_counts.items()))
        print(f"\n>>> Target Tag: [{cat_name}] | File: {fname} ({im_w}x{im_h}) | Boxes: {len(anns)} ({summary_str})")

        # Scale in-memory copy for clean notebook view if wide
        if im_w > max_display_width:
            scale = max_display_width / im_w
            display_img = img.resize((max_display_width, int(im_h * scale)), Image.BILINEAR)
        else:
            display_img = img

        # Directly print/display in notebook (Zero files saved to disk)
        display(display_img)

    print("\n" + "=" * 75)
    print("All category tag samples displayed inline successfully.")
    print("=" * 75)

# Aliases for compatibility with earlier calls
visualize_samples = lambda *args, **kwargs: print_sample_images_per_tag(TRAIN_ANN, str(IMAGES_DIR))
print_and_visualize_samples = visualize_samples

print_sample_images_per_tag(TRAIN_ANN, str(IMAGES_DIR))


In [ ]:
# CELL 14 - Safe Pretrained Weights Loading & Detection Head Patch
from rfdetr.models.lwdetr import LWDETR

# 1. Patch LWDETR.load_state_dict using torch.nn.Module.load_state_dict directly (prevents RecursionError)
def _safe_lwdetr_load_state_dict(self, state_dict, strict=True):
    """Filters out classification heads (class_embed, enc_out_class_embed) from checkpoint when num_classes differs."""
    model_state = self.state_dict()
    filtered_state_dict = {}
    mismatched = []
    
    for k, v in state_dict.items():
        clean_k = k
        if clean_k not in model_state and clean_k.startswith("model."):
            clean_k = clean_k[6:]
        if clean_k not in model_state and clean_k.startswith("module."):
            clean_k = clean_k[7:]
            
        if clean_k in model_state:
            if model_state[clean_k].shape == v.shape:
                filtered_state_dict[clean_k] = v
            else:
                mismatched.append((clean_k, tuple(v.shape), tuple(model_state[clean_k].shape)))
        elif k in model_state:
            if model_state[k].shape == v.shape:
                filtered_state_dict[k] = v
            else:
                mismatched.append((k, tuple(v.shape), tuple(model_state[k].shape)))

    if mismatched:
        logger.info(f"Notice: Filtered {len(mismatched)} classification head layers with class-count mismatch from COCO weights:")
        for name, ckpt_shape, model_shape in mismatched[:4]:
            logger.info(f"   - {name}: checkpoint {ckpt_shape} -> target model {model_shape}")
        if len(mismatched) > 4:
            logger.info(f"   - ... and {len(mismatched) - 4} more classification head tensors.")
        logger.info("Pretrained backbone, transformer, and regression heads loaded successfully.")
    
    # Call base PyTorch nn.Module.load_state_dict directly to avoid any recursion on re-runs
    return torch.nn.Module.load_state_dict(self, filtered_state_dict, strict=False)

LWDETR.load_state_dict = _safe_lwdetr_load_state_dict
logger.info("LWDETR.load_state_dict patched for safe custom num_classes loading.")

# 2. Patch reinitialize_detection_head to handle both class_embed and transformer.enc_out_class_embed
def _fixed_reinitialize(self, num_classes):
    lwdetr = self.model  # the actual LWDETR nn.Module
    device = next(lwdetr.parameters()).device
    
    # Reinitialize class_embed
    if hasattr(lwdetr, "class_embed"):
        in_feat = lwdetr.class_embed.in_features
        lwdetr.class_embed = nn.Linear(in_feat, num_classes).to(device)
        nn.init.normal_(lwdetr.class_embed.weight, std=0.01)
        nn.init.zeros_(lwdetr.class_embed.bias)
        
    # Reinitialize transformer.enc_out_class_embed
    if hasattr(lwdetr, "transformer") and hasattr(lwdetr.transformer, "enc_out_class_embed"):
        enc_heads = lwdetr.transformer.enc_out_class_embed
        if isinstance(enc_heads, nn.ModuleList):
            for i in range(len(enc_heads)):
                in_feat = enc_heads[i].in_features
                enc_heads[i] = nn.Linear(in_feat, num_classes).to(device)
                nn.init.normal_(enc_heads[i].weight, std=0.01)
                nn.init.zeros_(enc_heads[i].bias)
        elif isinstance(enc_heads, nn.Linear):
            in_feat = enc_heads.in_features
            lwdetr.transformer.enc_out_class_embed = nn.Linear(in_feat, num_classes).to(device)
            nn.init.normal_(lwdetr.transformer.enc_out_class_embed.weight, std=0.01)
            nn.init.zeros_(lwdetr.transformer.enc_out_class_embed.bias)
            
    logger.info(f"Detection heads (class_embed + enc_out_class_embed) initialized for num_classes={num_classes}")

rfdetr_main.Model.reinitialize_detection_head = _fixed_reinitialize
logger.info("reinitialize_detection_head patched.")

# 3. Model Instantiation
model_cls_map = {
    "nano": RFDETRNano,
    "small": RFDETRSmall,
    "medium": RFDETRMedium,
    "base": RFDETRBase,
    "large": RFDETRLarge
}
ModelClass = model_cls_map.get(MODEL_SIZE.lower(), RFDETRBase)
model = ModelClass(
    num_classes=NUM_CLASSES,
    pretrain_weights=PRETRAINED_WEIGHTS,
    resolution=RESOLUTION
)
logger.info(f"RF-DETR-{MODEL_SIZE.capitalize()} instantiated with resolution={RESOLUTION}, num_classes={NUM_CLASSES}")

# Explicitly ensure detection heads match NUM_CLASSES
if hasattr(model, "model") and hasattr(model.model, "reinitialize_detection_head"):
    model.model.reinitialize_detection_head(NUM_CLASSES)

# Build ordered list of class names from COCO categories
class_names = [
    train_info["categories"][cid]
    for cid in sorted(train_info["categories"].keys())
]
logger.info(f"class_names: {class_names}")

In [ ]:
# CELL 15 - Optimized PyTorch Dataset with Training Augmentations (Fast OpenCV C++ Decoding)
class COCODetectionDataset(Dataset):
    """Fast COCO dataset using OpenCV with optional data augmentations for training regularization."""
    def __init__(self, img_dir: str, ann_file: str, resolution: int = 560, is_train: bool = False, use_augmentation: bool = False):
        with open(ann_file) as f:
            data = json.load(f)
        self.img_dir = img_dir
        self.resolution = resolution
        self.is_train = is_train
        self.use_augmentation = use_augmentation and is_train
        self.cat_to_id = {c: i for i, c in enumerate(sorted(c["id"] for c in data["categories"]))}
        ann_by_img: dict = {}
        for ann in data["annotations"]:
            ann_by_img.setdefault(ann["image_id"], []).append(ann)
        self.samples = [(img, ann_by_img[img["id"]]) for img in data["images"] if img["id"] in ann_by_img]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_meta, anns = self.samples[idx]
        img_path = os.path.join(self.img_dir, img_meta["file_name"])

        # High-performance OpenCV decode & resize (3x faster than PIL on CPU)
        cv_img = cv2.imread(img_path)
        if cv_img is not None:
            orig_h, orig_w = cv_img.shape[:2]
            # Step C Augmentation: Random Horizontal Flip during training
            do_hflip = self.use_augmentation and (random.random() < 0.5)
            if do_hflip:
                cv_img = cv2.flip(cv_img, 1)

            # Step C Augmentation: Mild Color/Brightness jitter
            if self.use_augmentation and (random.random() < 0.5):
                alpha = 1.0 + random.uniform(-0.15, 0.15)  # Contrast
                beta = random.uniform(-15, 15)             # Brightness
                cv_img = np.clip(alpha * cv_img + beta, 0, 255).astype(np.uint8)

            resized = cv2.resize(cv_img, (self.resolution, self.resolution), interpolation=cv2.INTER_LINEAR)
            rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
            img_t = torch.from_numpy(rgb).permute(2, 0, 1).float().div_(255.0)
        else:
            with Image.open(img_path).convert("RGB") as pil_im:
                orig_w, orig_h = pil_im.size
                do_hflip = self.use_augmentation and (random.random() < 0.5)
                if do_hflip:
                    pil_im = TF.hflip(pil_im)
                resized = pil_im.resize((self.resolution, self.resolution), Image.BILINEAR)
                img_t = TF.to_tensor(resized)

        img_t = TF.normalize(img_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0: continue
            
            cx = (x + w / 2) / orig_w
            cy = (y + h / 2) / orig_h
            bw = w / orig_w
            bh = h / orig_h
            if do_hflip:
                cx = 1.0 - cx

            boxes.append([
                float(np.clip(cx, 0, 1)),
                float(np.clip(cy, 0, 1)),
                float(np.clip(bw, 0, 1)),
                float(np.clip(bh, 0, 1))
            ])
            labels.append(self.cat_to_id[ann["category_id"]])

        return img_t, {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            "labels": torch.tensor(labels, dtype=torch.long) if labels else torch.zeros(0, dtype=torch.long),
            "image_id": torch.tensor([img_meta["id"]]),
            "orig_size": torch.tensor([orig_h, orig_w]),
            "size": torch.tensor([self.resolution, self.resolution]),
        }

def collate_fn(batch):
    return torch.stack([item[0] for item in batch]), [item[1] for item in batch]

logger.info(f"Fast COCODetectionDataset ready with data augmentation support (is_train={USE_AUGMENTATION}).")

In [ ]:
# CELL 16 - Criterion, Asynchronous DataLoaders, AdamW & Step C Early Stopping Checkpointer
args = copy.deepcopy(model.model.args)
args.num_classes = NUM_CLASSES
args.device = "cuda" if torch.cuda.is_available() else "cpu"
criterion, _ = build_criterion_and_postprocessors(args)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lwdetr = model.model.model
lwdetr.to(device)
criterion.to(device)

# Step C: Instantiate datasets using master settings
train_ds = COCODetectionDataset(str(IMAGES_DIR), TRAIN_ANN, RESOLUTION, is_train=True, use_augmentation=USE_AUGMENTATION)
val_ds = COCODetectionDataset(str(IMAGES_DIR), VAL_ANN, RESOLUTION, is_train=False, use_augmentation=False)

# High-throughput asynchronous DataLoaders configured from Master Settings
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    drop_last=True,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0 and PERSISTENT_WORKERS),
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
)
val_loader = DataLoader(
    val_ds,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0 and PERSISTENT_WORKERS),
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
)

# Backbone Freezing Control for Massive Speedup
if FREEZE_BACKBONE and hasattr(lwdetr, "backbone"):
    for p in lwdetr.backbone.parameters():
        p.requires_grad = False
    logger.info("DINOv2 backbone frozen: training only DETR encoder/decoder & heads (~60% speedup).")
else:
    logger.info("Full model fine-tuning enabled (including DINOv2 backbone).")

trainable_params = [p for p in lwdetr.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

# Mixed precision GradScaler
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# Step C: Enhanced Checkpointer with Early Stopping, Training Resume & Full History Logging
# Precision-Aware Model Checkpointer tracking Metric Precision, F1, and mAP
class PrecisionAwareCheckpointSD:
    def __init__(
        self,
        checkpoint_dir: str,
        module: nn.Module,
        monitor_metric: str = MONITOR_METRIC,
        patience: int = EARLY_STOPPING_PATIENCE,
        stop_on_overfitting: bool = True,
        overfit_ratio_threshold: float = 2.5,
        overfit_patience: int = 2,
        warmup_epochs: int = 3,
        verbose: bool = True
    ):
        self.ckpt_dir = Path(checkpoint_dir); self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.module, self.verbose = module, verbose
        self.monitor_metric = monitor_metric.lower()
        self.patience = patience
        self.stop_on_overfitting = stop_on_overfitting
        self.overfit_ratio_threshold = overfit_ratio_threshold
        self.overfit_patience = overfit_patience
        self.warmup_epochs = warmup_epochs

        self.best_loss = float("inf")
        self.best_precision = 0.0
        self.best_recall = 0.0
        self.best_f1 = 0.0
        self.best_map50 = 0.0
        self.best_score = -float("inf")
        self.best_epoch = -1
        self.history = []

        self.consecutive_no_improve = 0
        self.consecutive_overfit = 0
        self.consecutive_divergence = 0
        self.should_stop = False
        self.stop_reason = ""
        self.log_path = self.ckpt_dir / "training_log.json"
        self.best_path = self.ckpt_dir / "best_model.pth"
        self.best_precision_path = self.ckpt_dir / "best_model_precision.pth"
        self.latest_path = self.ckpt_dir / "latest_checkpoint.pth"

    def step(
        self,
        epoch: int,
        train_loss: float,
        val_loss: float,
        optimizer=None,
        scheduler=None,
        scaler=None,
        train_comp: dict = None,
        val_comp: dict = None,
        val_precision: float = None,
        val_recall: float = None,
        val_f1: float = None,
        val_map50: float = None,
        val_map: float = None,
        class_precisions: dict = None
    ) -> bool:
        # Determine tracking score based on monitor_metric
        val_prec = val_precision if val_precision is not None else 0.0
        val_m50 = val_map50 if val_map50 is not None else 0.0
        val_f1_score = val_f1 if val_f1 is not None else 0.0

        if self.monitor_metric == "precision":
            current_score = val_prec
            improved = current_score > self.best_score
        elif self.monitor_metric == "f1":
            current_score = val_f1_score
            improved = current_score > self.best_score
        elif self.monitor_metric == "map_50":
            current_score = val_m50
            improved = current_score > self.best_score
        elif self.monitor_metric == "composite":
            current_score = (COMPOSITE_PRECISION_WEIGHT * val_prec) + ((1.0 - COMPOSITE_PRECISION_WEIGHT) * val_m50)
            improved = current_score > self.best_score
        else: # fallback to validation loss
            current_score = -val_loss
            improved = val_loss < self.best_loss

        # Track best loss regardless
        if val_loss < self.best_loss:
            self.best_loss = val_loss

        if improved:
            self.best_score = current_score
            self.best_epoch = epoch
            self.best_precision = max(self.best_precision, val_prec)
            self.best_recall = max(self.best_recall, val_recall if val_recall is not None else 0.0)
            self.best_f1 = max(self.best_f1, val_f1_score)
            self.best_map50 = max(self.best_map50, val_m50)
            self.consecutive_no_improve = 0

            # Save best general checkpoint and dedicated precision checkpoint
            torch.save(self.module.state_dict(), str(self.best_path))
            if val_precision is not None and val_prec >= self.best_precision:
                torch.save(self.module.state_dict(), str(self.best_precision_path))
            torch.save(self.module.state_dict(), str(self.ckpt_dir / f"epoch_{epoch:04d}_prec{val_prec:.4f}_map{val_m50:.4f}.pth"))
        else:
            self.consecutive_no_improve += 1
            if self.consecutive_no_improve >= self.patience:
                self.should_stop = True
                self.stop_reason = f"Metric ({self.monitor_metric}) failed to improve for {self.patience} consecutive epochs (Best {self.monitor_metric}: {self.best_score:.4f} at Epoch {self.best_epoch})."

        ratio = round(val_loss / max(train_loss, 1e-6), 2)

        # Overfitting check
        if self.stop_on_overfitting and epoch >= self.warmup_epochs:
            if ratio >= self.overfit_ratio_threshold and not improved:
                self.consecutive_overfit += 1
                if self.consecutive_overfit >= self.overfit_patience:
                    self.should_stop = True
                    self.stop_reason = f"Overfitting detected: Val/Train loss ratio reached {ratio:.1f}x without metric improvement for {self.consecutive_overfit} consecutive epochs."
            else:
                self.consecutive_overfit = 0

        entry = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_train_ratio": ratio,
            "val_precision": val_precision,
            "val_recall": val_recall,
            "val_f1": val_f1,
            "val_map50": val_map50,
            "val_map50_95": val_map,
            "class_precisions": class_precisions or {},
            "train_components": train_comp or {},
            "val_components": val_comp or {},
            "time": time.strftime("%Y-%m-%dT%H:%M:%S")
        }
        self.history.append(entry)

        # Save training log JSON
        with open(self.log_path, "w") as f:
            json.dump(self.history, f, indent=2)

        # Always save latest checkpoint
        torch.save({
            "epoch": epoch,
            "state_dict": self.module.state_dict(),
            "optimizer": optimizer.state_dict() if optimizer else None,
            "scheduler": scheduler.state_dict() if scheduler else None,
            "scaler": scaler.state_dict() if scaler else None,
            "best_score": self.best_score,
            "best_precision": self.best_precision,
            "best_epoch": self.best_epoch
        }, str(self.latest_path))

        return improved

    def load_best(self):
        """Loads the best model weights into the wrapped module."""
        load_target = self.best_precision_path if (self.monitor_metric == "precision" and self.best_precision_path.exists()) else self.best_path
        if load_target.exists():
            state = torch.load(str(load_target), map_location="cpu")
            self.module.load_state_dict(state)
            if self.verbose:
                logger.info(f"Loaded best checkpoint from: {load_target}")
        else:
            logger.warning(f"No checkpoint found at: {load_target}")

ckpt = PrecisionAwareCheckpointSD(CHECKPOINT_DIR, lwdetr, monitor_metric=MONITOR_METRIC, patience=EARLY_STOPPING_PATIENCE)
logger.info(f"DataLoaders: Train batches={len(train_loader)} (B={BATCH_SIZE}), Val batches={len(val_loader)} (B={VAL_BATCH_SIZE})")
print_vram_usage("Pre-Training")

In [ ]:
# CELL 17 - Optimized Training Step (AMP, Accumulation) & Precision Metric Tracking
scaler = globals().get("scaler", torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available()))

from collections import defaultdict
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def box_iou_pure_torch(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    """Calculates pairwise IoU between two sets of boxes [x1, y1, x2, y2] using pure PyTorch."""
    if len(boxes1) == 0 or len(boxes2) == 0:
        return torch.zeros((len(boxes1), len(boxes2)), device=boxes1.device if len(boxes1) > 0 else boxes2.device)
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)
    lt = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area1[:, None] + area2[None, :] - inter
    return inter / union.clamp(min=1e-7)

def compute_precision_recall_f1(preds_list: list, tgts_list: list, iou_thresh: float = 0.50, conf_thresh: float = 0.50) -> dict:
    """
    Computes exact operational Detection Precision, Recall, F1-Score, and per-class Precision
    at specified confidence and IoU thresholds.
    """
    total_tp, total_fp, total_fn = 0, 0, 0
    per_class_stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

    for pred, tgt in zip(preds_list, tgts_list):
        p_boxes = pred["boxes"]
        p_scores = pred["scores"]
        p_labels = pred["labels"]

        keep = p_scores >= conf_thresh
        p_boxes = p_boxes[keep]
        p_scores = p_scores[keep]
        p_labels = p_labels[keep]

        t_boxes = tgt["boxes"]
        t_labels = tgt["labels"]

        if len(p_boxes) == 0:
            total_fn += len(t_boxes)
            for tl in t_labels.tolist():
                per_class_stats[tl]["fn"] += 1
            continue

        if len(t_boxes) == 0:
            total_fp += len(p_boxes)
            for pl in p_labels.tolist():
                per_class_stats[pl]["fp"] += 1
            continue

        ious = box_iou_pure_torch(p_boxes, t_boxes)
        matched_tgt = set()

        # Match highest-confidence predictions first
        sort_idx = torch.argsort(p_scores, descending=True)
        for p_idx in sort_idx.tolist():
            p_cat = p_labels[p_idx].item()
            best_iou, best_t_idx = -1.0, -1
            for t_idx in range(len(t_boxes)):
                if t_idx in matched_tgt:
                    continue
                if t_labels[t_idx].item() == p_cat:
                    iou = ious[p_idx, t_idx].item()
                    if iou > best_iou and iou >= iou_thresh:
                        best_iou, best_t_idx = iou, t_idx

            if best_t_idx >= 0:
                matched_tgt.add(best_t_idx)
                total_tp += 1
                per_class_stats[p_cat]["tp"] += 1
            else:
                total_fp += 1
                per_class_stats[p_cat]["fp"] += 1

        for t_idx in range(len(t_boxes)):
            if t_idx not in matched_tgt:
                total_fn += 1
                t_cat = t_labels[t_idx].item()
                per_class_stats[t_cat]["fn"] += 1

    precision = total_tp / max(1, total_tp + total_fp)
    recall = total_tp / max(1, total_tp + total_fn)
    f1 = 2 * precision * recall / max(1e-6, precision + recall)

    class_precisions = {}
    for cat_id, st in per_class_stats.items():
        cat_name = ID_TO_CATEGORY.get(cat_id, f"class_{cat_id}")
        c_prec = st["tp"] / max(1, st["tp"] + st["fp"])
        class_precisions[cat_name] = round(c_prec, 4)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn,
        "class_precisions": class_precisions
    }

def run_epoch(model, loader, criterion, optimizer, device, is_train: bool, epoch: int, num_epochs: int, max_steps: Optional[int] = None):
    """Runs one training or validation epoch with Step A loss breakdown tracking."""
    model.train(is_train)
    phase = "train" if is_train else "val"
    total, count = 0.0, 0
    component_totals = defaultdict(float)
    total_steps = min(len(loader), max_steps) if (is_train and max_steps) else len(loader)
    pbar = tqdm(loader, total=total_steps, desc=f"Epoch {epoch:>2}/{num_epochs} [{phase}]", leave=False, unit="batch", dynamic_ncols=True)

    with torch.set_grad_enabled(is_train):
        for step, (images, targets) in enumerate(pbar):
            if is_train and max_steps and (step >= max_steps):
                break
            images = images.to(device, non_blocking=True)
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss_dict = criterion(outputs, targets)
                loss = sum(loss_dict[k] * criterion.weight_dict[k] for k in loss_dict if k in criterion.weight_dict)

            if is_train:
                loss_scaled = loss / GRAD_ACCUMULATION
                scaler.scale(loss_scaled).backward()
                if (step + 1) % GRAD_ACCUMULATION == 0 or (step + 1) == len(loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

            total += loss.item()
            count += 1
            for k, v in loss_dict.items():
                if k in criterion.weight_dict:
                    component_totals[k] += (v.item() * criterion.weight_dict[k])

    avg_loss = total / max(count, 1)
    avg_components = {k: v / max(count, 1) for k, v in component_totals.items()}
    return avg_loss, avg_components

def recalibrate_and_average_boxes(
    boxes: list,
    scores: list,
    class_ids: list,
    enable_averaging: bool = True,
    enable_shrinkage: bool = True,
    shrink_factor: float = 0.96,
    fusion_iou: float = 0.50,
    max_w: float = 1e6,
    max_h: float = 1e6
) -> list:
    """
    Recalibrate detections via:
    1. Boundary-Box Shrinkage: Contracts loose query boundaries towards center by shrink_factor
       (e.g., 0.96 shrinks width and height by 4%), eliminating background margin jitter
       and tightening crops for downstream PaddleOCR.
    2. Bounding-Box Averaging: Clusters overlapping queries of the same class (IoU >= fusion_iou)
       and performs confidence-weighted coordinate averaging, suppressing duplicate false alarms.
    """
    if not boxes or len(boxes) == 0:
        return []
    
    # 1. Boundary-box shrinkage
    shrunk = []
    for i, b in enumerate(boxes):
        if enable_shrinkage and shrink_factor < 1.0:
            cx = (b[0] + b[2]) / 2.0
            cy = (b[1] + b[3]) / 2.0
            w = (b[2] - b[0]) * shrink_factor
            h = (b[3] - b[1]) * shrink_factor
            x1 = max(0.0, cx - w / 2.0)
            y1 = max(0.0, cy - h / 2.0)
            x2 = min(float(max_w), cx + w / 2.0)
            y2 = min(float(max_h), cy + h / 2.0)
            shrunk_b = [x1, y1, x2, y2]
        else:
            shrunk_b = list(b)
        
        shrunk.append({
            "box": shrunk_b,
            "score": float(scores[i]),
            "class_id": int(class_ids[i])
        })
        
    if not enable_averaging or len(shrunk) <= 1:
        return shrunk
        
    # 2. Bounding-box weighted averaging (WBF)
    fused = []
    distinct_classes = set(p["class_id"] for p in shrunk)
    for cid in distinct_classes:
        c_preds = [p for p in shrunk if p["class_id"] == cid]
        c_preds.sort(key=lambda x: x["score"], reverse=True)
        
        clusters = []
        for p in c_preds:
            matched = False
            for cl in clusters:
                b1, b2 = p["box"], cl[0]["box"]
                xA, yA = max(b1[0], b2[0]), max(b1[1], b2[1])
                xB, yB = min(b1[2], b2[2]), min(b1[3], b2[3])
                inter = max(0.0, xB - xA) * max(0.0, yB - yA)
                a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
                a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
                iou = inter / max(1e-7, a1 + a2 - inter)
                if iou >= fusion_iou:
                    cl.append(p)
                    matched = True
                    break
            if not matched:
                clusters.append([p])
                
        for cl in clusters:
            if len(cl) == 1:
                fused.append(cl[0])
            else:
                weights = [max(0.01, c["score"]) for c in cl]
                total_w = sum(weights)
                avg_x1 = sum(c["box"][0] * w for c, w in zip(cl, weights)) / total_w
                avg_y1 = sum(c["box"][1] * w for c, w in zip(cl, weights)) / total_w
                avg_x2 = sum(c["box"][2] * w for c, w in zip(cl, weights)) / total_w
                avg_y2 = sum(c["box"][3] * w for c, w in zip(cl, weights)) / total_w
                consensus_score = min(1.0, max(c["score"] for c in cl) + 0.01 * (len(cl) - 1))
                fused.append({
                    "box": [avg_x1, avg_y1, avg_x2, avg_y2],
                    "score": consensus_score,
                    "class_id": cid,
                    "fused_count": len(cl)
                })
    fused.sort(key=lambda x: x["score"], reverse=True)
    return fused

def compute_detection_metrics(model, loader, device, img_size: int, conf_thresh: float = TARGET_PRECISION_CONF_THRESH, iou_thresh: float = TARGET_PRECISION_IOU_THRESH) -> dict:
    """
    Computes comprehensive validation metrics:
    - Standard COCO mAP (mAP@50, mAP@75, mAP@50:95, mAR@100)
    - Operational Precision, Recall, and F1 at target confidence threshold
    - Per-class Precision breakdown
    """
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True, max_detection_thresholds=[1, 10, 300])
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
            pred_logits, pred_boxes = outputs["pred_logits"], outputs["pred_boxes"]
            
            for i in range(len(images)):
                scores, lbs = pred_logits[i].sigmoid().max(-1)
                p_dict = {
                    "boxes": (box_cxcywh_to_xyxy(pred_boxes[i]) * img_size).cpu().float(),
                    "scores": scores.cpu().float(),
                    "labels": lbs.cpu(),
                }
                t_dict = {
                    "boxes": (box_cxcywh_to_xyxy(targets[i]["boxes"]) * img_size).cpu().float(),
                    "labels": targets[i]["labels"].cpu(),
                }
                all_preds.append(p_dict)
                all_targets.append(t_dict)
                
                # Filter for mAP calculation
                keep = scores > MAP_EVAL_THRESHOLD
                metric.update(
                    [{
                        "boxes": p_dict["boxes"][keep],
                        "scores": p_dict["scores"][keep],
                        "labels": p_dict["labels"][keep]
                    }],
                    [t_dict]
                )

    map_res = metric.compute()
    prec_res = compute_precision_recall_f1(all_preds, all_targets, iou_thresh=iou_thresh, conf_thresh=conf_thresh)

    def _to_f(key):
        val = map_res.get(key, 0.0)
        return float(val.item()) if hasattr(val, "item") else float(val)

    return {
        "precision": prec_res["precision"],
        "recall": prec_res["recall"],
        "f1": prec_res["f1"],
        "class_precisions": prec_res["class_precisions"],
        "tp": prec_res["tp"],
        "fp": prec_res["fp"],
        "fn": prec_res["fn"],
        "map_50": _to_f("map_50"),
        "map_75": _to_f("map_75"),
        "map": _to_f("map"),
        "mar_100": _to_f("mar_100")
    }

logger.info("Compiled optimized training step and compute_detection_metrics with Precision, Recall, F1 and mAP.")


In [ ]:
# CELL 18 - Precision-Driven Training Loop: Tracking Metric Precision, F1, Loss & mAP
# 1. Resume Checkpoint Check
if RESUME_TRAINING:
    start_epoch = ckpt.try_resume(optimizer=optimizer, scheduler=scheduler, scaler=scaler, checkpoint_path=RESUME_CHECKPOINT_PATH)
else:
    start_epoch = 1

if start_epoch > EPOCHS:
    logger.info("=" * 95)
    logger.info(f"Training already completed up to Epoch {start_epoch - 1} (target EPOCHS={EPOCHS}).")
    logger.info("=" * 95)
else:
    logger.info("=" * 95)
    logger.info(f"Precision-Driven Training Started: Epochs {start_epoch} -> {EPOCHS} | Batch: {BATCH_SIZE} (Eff={EFFECTIVE_BATCH_SIZE}) | Res: {RESOLUTION}x{RESOLUTION}")
    logger.info(f"Monitoring Metric: {MONITOR_METRIC.upper()} (Target Conf >= {TARGET_PRECISION_CONF_THRESH}, IoU >= {TARGET_PRECISION_IOU_THRESH})")
    logger.info("=" * 95)

    for epoch in range(start_epoch, EPOCHS + 1):
        t_start = time.time()
        tr_loss, tr_comp = run_epoch(lwdetr, train_loader, criterion, optimizer, device, is_train=True, epoch=epoch, num_epochs=EPOCHS, max_steps=STEPS_PER_EPOCH)
        val_loss, val_comp = run_epoch(lwdetr, val_loader, criterion, optimizer, device, is_train=False, epoch=epoch, num_epochs=EPOCHS)
        
        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        # Compute complete validation metrics (Precision, Recall, F1, mAP@50, mAP@50:95)
        metrics = compute_detection_metrics(
            lwdetr, val_loader, device,
            img_size=RESOLUTION,
            conf_thresh=TARGET_PRECISION_CONF_THRESH,
            iou_thresh=TARGET_PRECISION_IOU_THRESH
        )

        val_prec = metrics["precision"]
        val_rec  = metrics["recall"]
        val_f1   = metrics["f1"]
        val_map50 = metrics["map_50"]
        val_map   = metrics["map"]
        class_precs = metrics["class_precisions"]

        # Step checkpointer with precision awareness
        improved = ckpt.step(
            epoch=epoch,
            train_loss=tr_loss,
            val_loss=val_loss,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            train_comp=tr_comp,
            val_comp=val_comp,
            val_precision=val_prec,
            val_recall=val_rec,
            val_f1=val_f1,
            val_map50=val_map50,
            val_map=val_map,
            class_precisions=class_precs
        )

        tag = f" <-- [BEST {MONITOR_METRIC.upper()} MODEL SAVED: {val_prec:.4f}]" if improved else ""
        elapsed = time.time() - t_start
        time_str = f"{int(elapsed // 60)}m {int(elapsed % 60):02d}s"

        tr_sub = f"cls={tr_comp.get('loss_ce', 0):.2f}, box={tr_comp.get('loss_bbox', 0):.2f}, giou={tr_comp.get('loss_giou', 0):.2f}"
        val_sub = f"cls={val_comp.get('loss_ce', 0):.2f}, box={val_comp.get('loss_bbox', 0):.2f}, giou={val_comp.get('loss_giou', 0):.2f}"

        # 4-line diagnostic status with live Precision metrics
        logger.info(f"Epoch {epoch:02d}/{EPOCHS} [{time_str}] | LR: {current_lr:.1e} | Best {MONITOR_METRIC.upper()}: {ckpt.best_score:.4f}{tag}")
        logger.info(f"  • Loss      -> Train: {tr_loss:.4f} ({tr_sub}) | Val: {val_loss:.4f} ({val_sub})")
        logger.info(f"  • Metrics   -> Precision: {val_prec:.4f} | Recall: {val_rec:.4f} | F1: {val_f1:.4f} | mAP@50: {val_map50:.4f} | mAP@50:95: {val_map:.4f}")
        class_prec_str = ", ".join([f"{k}: {v:.4f}" for k, v in class_precs.items()]) if class_precs else "N/A"
        logger.info(f"  • Per-Class -> {class_prec_str}")

        if ckpt.should_stop:
            logger.info("=" * 95)
            logger.info(f"[EARLY STOPPING TRIGGERED] Halting training at Epoch {epoch}.")
            logger.info(f"  • Reason: {ckpt.stop_reason}")
            logger.info(f"  • Restoring Best Checkpoint from Epoch {ckpt.best_epoch} with Best {MONITOR_METRIC.upper()}: {ckpt.best_score:.4f}")
            logger.info("=" * 95)
            ckpt.load_best()
            break

    logger.info("=" * 95)
    logger.info(f"Training Finished! Best {MONITOR_METRIC.upper()}: {ckpt.best_score:.4f} (Precision: {ckpt.best_precision:.4f}, mAP@50: {ckpt.best_map50:.4f}) at Epoch {ckpt.best_epoch}.")
    logger.info("=" * 95)


In [ ]:
# CELL 19 - Load Best Precision Checkpoint & Final Validation Benchmark
logger.info("=" * 70)
logger.info("[CELL 19] Loading best precision checkpoint and running validation audit...")
logger.info("=" * 70)

ckpt.load_best()
final_metrics = compute_detection_metrics(
    lwdetr, val_loader, device,
    img_size=RESOLUTION,
    conf_thresh=TARGET_PRECISION_CONF_THRESH,
    iou_thresh=TARGET_PRECISION_IOU_THRESH
)

logger.info("=" * 60)
logger.info("Final Validation Benchmark (Best Model):")
logger.info(f"   - Metric Precision:    {final_metrics['precision']:.4f} (TP={final_metrics['tp']}, FP={final_metrics['fp']})")
logger.info(f"   - Metric Recall:       {final_metrics['recall']:.4f} (FN={final_metrics['fn']})")
logger.info(f"   - Metric F1-Score:     {final_metrics['f1']:.4f}")
logger.info(f"   - COCO mAP@50:         {final_metrics['map_50']:.4f}")
logger.info(f"   - COCO mAP@75:         {final_metrics['map_75']:.4f}")
logger.info(f"   - COCO mAP@50:95:      {final_metrics['map']:.4f}")
logger.info(f"   - COCO mAR@100:        {final_metrics['mar_100']:.4f}")
logger.info("-" * 60)
logger.info("Per-Class Precision Breakdown:")
for cat_name, c_prec in final_metrics.get("class_precisions", {}).items():
    logger.info(f"   - {cat_name:20s}: {c_prec:.4f}")
logger.info("=" * 60)


In [ ]:
# CELL 20 - Export Best Precision Model to model/ Directory
logger.info("=" * 70)
logger.info("[CELL 20] Exporting best model checkpoint for deployment & predictions...")
logger.info("=" * 70)

# Export primary best_model.pth and best_model_precision.pth
export_targets = [
    Path(FINAL_MODEL_DIR) / "rfdetr_best.pth",
    Path(FINAL_MODEL_DIR) / "rfdetr_best_precision.pth",
    Path(CHECKPOINT_DIR) / "best_model.pth"
]

for tgt in export_targets:
    tgt.parent.mkdir(parents=True, exist_ok=True)
    torch.save(lwdetr.state_dict(), str(tgt))
    logger.info(f"Exported model weights -> {tgt} ({os.path.getsize(tgt) / (1024*1024):.1f} MB)")

# Save metadata manifest
metadata = {
    "model_name": "RF-DETR Base Clean Multi-Class",
    "version": MODEL_VERSION,
    "run_id": RUN_ID,
    "categories": TARGET_CATEGORIES,
    "resolution": RESOLUTION,
    "best_epoch": ckpt.best_epoch,
    "monitor_metric": MONITOR_METRIC,
    "metrics": {
        "precision": final_metrics["precision"],
        "recall": final_metrics["recall"],
        "f1": final_metrics["f1"],
        "map_50": final_metrics["map_50"],
        "map_50_95": final_metrics["map"],
        "class_precisions": final_metrics["class_precisions"]
    }
}

manifest_path = Path(FINAL_MODEL_DIR) / "model_metadata.json"
with open(manifest_path, "w") as f:
    json.dump(metadata, f, indent=2)

logger.info(f"Saved model metadata -> {manifest_path}")


In [ ]:
# CELL 21 - Test Set Inference & Visual Predictions (Pure PIL & IPython - Zero Matplotlib Dependency)
from PIL import Image
from IPython.display import Image as IPImage, display

logger.info("=" * 70)
logger.info(f"[CELL 21] Running inference on test split (Confidence={CONFIDENCE}, NMS={NMS_THRESHOLD})...")
logger.info("=" * 70)

import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass
try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

test_ds = COCODetectionDataset(str(IMAGES_DIR), TEST_ANN, RESOLUTION)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

test_map_results = compute_map(lwdetr, test_loader, device, img_size=RESOLUTION, threshold=CONFIDENCE_THRESHOLD)
logger.info("Test Set mAP Results:")
for k, v in test_map_results.items():
    if isinstance(v, torch.Tensor):
        if v.numel() == 1:
            logger.info(f"   - {k:20s}: {v.item():.4f}")
        else:
            logger.info(f"   - {k:20s}: {[round(x, 4) for x in v.tolist()]}")

# Visual sample test predictions using supervision + PIL save/display
color_palette = sv.ColorPalette.from_hex([
    "#e6194B", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990"
])
box_annotator = sv.BoxAnnotator(color=color_palette, thickness=2)
label_annotator = sv.LabelAnnotator(color=color_palette, text_scale=0.5, text_thickness=1)

os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)
lwdetr.eval()
preview_count = 0

with torch.no_grad():
    for images, targets in test_loader:
        if preview_count >= NUM_INFERENCE_PREVIEWS:
            break
        images = images.to(device)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            outputs = lwdetr(images)
        pred_logits = outputs["pred_logits"][0]
        pred_boxes = outputs["pred_boxes"][0]

        scores, class_ids = pred_logits.sigmoid().max(-1)
        keep = scores > CONFIDENCE
        if keep.sum() == 0:
            continue

        boxes_xyxy = (box_cxcywh_to_xyxy(pred_boxes[keep]) * RESOLUTION).cpu().numpy()
        scores_np = scores[keep].cpu().numpy()
        class_ids_np = class_ids[keep].cpu().numpy()

        img_np = (images[0].permute(1, 2, 0).cpu().numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)

        detections = sv.Detections(xyxy=boxes_xyxy, confidence=scores_np, class_id=class_ids_np)
        labels = [f"{class_names[cid] if cid < len(class_names) else str(cid)} {c:.2f}" for cid, c in zip(class_ids_np, scores_np)]
        annotated = box_annotator.annotate(scene=img_np.copy(), detections=detections)
        annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

        # Save annotated image directly and display via IPython (no Matplotlib)
        out_img_path = os.path.join(INFERENCE_OUTPUT_DIR, f"preview_detection_{preview_count+1}.png")
        Image.fromarray(annotated).save(out_img_path)
        logger.info(f"Preview #{preview_count+1}: {len(boxes_xyxy)} objects detected -> Saved to {out_img_path}")
        display(IPImage(filename=out_img_path))
        preview_count += 1

logger.info(f"Successfully generated and previewed {preview_count} test detection images.")


In [ ]:
# CELL 22 - Save Test Predictions JSON
logger.info("=" * 70)
logger.info("[CELL 22] Saving test predictions...")
logger.info("=" * 70)

pred_out_file = os.path.join(INFERENCE_OUTPUT_DIR, f"test_predictions_{RUN_ID}.json")
lwdetr.eval()
all_test_predictions = {}

with torch.no_grad():
    for images, targets in test_loader:
        img_id = targets[0]["image_id"].item()
        orig_h, orig_w = targets[0]["orig_size"].tolist()
        images = images.to(device)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            outputs = lwdetr(images)
        pred_logits = outputs["pred_logits"][0]
        pred_boxes = outputs["pred_boxes"][0]

        scores, class_ids = pred_logits.sigmoid().max(-1)
        keep = scores > CONFIDENCE

        boxes_xyxy = (box_cxcywh_to_xyxy(pred_boxes[keep]).cpu().numpy() * np.array([orig_w, orig_h, orig_w, orig_h])).tolist()
        scores_list = scores[keep].cpu().tolist()
        class_ids_list = class_ids[keep].cpu().tolist()

        all_test_predictions[str(img_id)] = {
            "boxes_xyxy": boxes_xyxy,
            "confidence": scores_list,
            "class_id": class_ids_list,
            "orig_size": [orig_h, orig_w]
        }

with open(pred_out_file, "w", encoding="utf-8") as f:
    json.dump({
        "pipeline": PIPELINE_NAME,
        "mode": MODE_TAG,
        "num_classes": NUM_CLASSES,
        "class_names": class_names,
        "confidence_threshold": CONFIDENCE,
        "predictions": all_test_predictions
    }, f, indent=2)



In [ ]:
# CELL 23 - Final Execution Summary
logger.info("=" * 70)
logger.info(f"[CELL 23] Execution Summary: {PIPELINE_NAME}")
logger.info("=" * 70)
logger.info(f"   - Model Version:        {MODEL_VERSION} (Run ID: {RUN_ID})")
logger.info(f"   - Pipeline Mode:        {MODE_TAG.upper()}")
logger.info(f"   - Number of Classes:    {NUM_CLASSES} ({class_names})")
logger.info(f"   - Architecture:         RF-DETR-{MODEL_SIZE.capitalize()} (resolution={RESOLUTION})")
logger.info(f"   - Effective Batch Size: {BATCH_SIZE * GRAD_ACCUMULATION} (Batch={BATCH_SIZE}, Accum={GRAD_ACCUMULATION})")
logger.info(f"   - Learning Rate:        {LEARNING_RATE} (Weight Decay: {WEIGHT_DECAY})")
logger.info(f"   - Dataset Folder:       {DATASET_DIR}")
logger.info(f"   - Images Folder:        {IMAGES_DIR} (Zero-copy shared repository)")
logger.info(f"   - Checkpoint Folder:    {CHECKPOINT_DIR}")
logger.info(f"   - Exported Model:       {exported_path}")
logger.info(f"   - Test Predictions:     {pred_out_file}")
logger.info(f"   - Pipeline Log File:    {log_file_path}")
logger.info("=" * 70)
